# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s0m-a/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# Task type: Binary Classification (with a future path to scoring/ranking)

***My lane is Lane 1***: Ranking Signal Analysis. The ML task is classification, given a page's observable signals today (position, impressions, CTR, content age), predict whether it is currently declining in traffic (is_declining_label = True) or not.

I considered multiclass (Declining / Stable / Growing) but the starter dataset provides a reliable binary label derived from trend_direction. Starting binary keeps the target honest and directly observed. The output of classification feeds a downstream priority queue — pages predicted as declining get ranked by model confidence and surfaced first for editorial review.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target: is_declining_label (binary: True = declining, False = stable or improving)

***Where it comes from***: This is an observed outcome, not a rule I invented. It is derived from trend_direction == "down", which is measured from real search performance data over a 90-day window. This makes it a legitimate ML target — the model learns from what actually happened to pages in the data, not from a hand-written score.

***The leakage risk***: trend_direction and trend_pct cannot be used as features because the label is derived from them. I will exclude them from all feature sets.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Confirm the target column exists and show its distribution
df["is_declining_label"] = (df["trend_direction"] == "down")
print("Target distribution:")
print(df["is_declining_label"].value_counts())
print(f"\nDecline rate: {df['is_declining_label'].mean():.1%}")


Target distribution:
is_declining_label
True     16262
False    13738
Name: count, dtype: int64

Decline rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Primary metric: Precision@50

An editor can realistically review 50 pages per week. "Good" means: **of the top 50 pages my model flags as most urgently declining, what fraction are actually declining?**

***Baseline (hand rule)***: 0.62 Precision@50
***Target***: Beat 0.62 — the model earns its place only if it outperforms a simple rule

I use Precision@50 over accuracy because the dataset is imbalanced and the real-world constraint is editorial bandwidth, not prediction volume.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show baseline rate to confirm metric makes sense
total = len(df)
declining = df["is_declining_label"].sum()
print(f"Total pages: {total:,}")
print(f"Declining pages: {declining:,} ({declining/total:.1%})")
print(f"\nA random baseline Precision@50 would be ~{declining/total:.2f}")
print("Our hand-rule baseline achieved 0.62 — that is the bar to beat.")


Total pages: 30,000
Declining pages: 16,262 (54.2%)

A random baseline Precision@50 would be ~0.54
Our hand-rule baseline achieved 0.62 — that is the bar to beat.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

***One row = one pseudonymised content page, observed over a 90-day window.***

Each row represents a single URL aggregated across 90 days of search activity. It is not a daily snapshot. The key columns for my lane are: avg_position, impressions_90d, ctr, content_age_days, word_count, and is_declining_label.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_cols = [
    "avg_position", "impressions_90d", "ctr",
    "content_age_days", "word_count",
    "trend_direction", "is_declining_label"
]
print(f"Shape: {df.shape}")
print(f"\nOne row = one page over 90 days\n")
df[lane_cols].head(5)



Shape: (30000, 45)

One row = one page over 90 days



,avg_position,impressions_90d,ctr,content_age_days,word_count,trend_direction,is_declining_label
0,10.6,3803,0.76,187,3221.0,down,True
1,20.3,15320,0.05,445,2481.0,down,True
2,36.5,12581,0.09,141,3515.0,down,True
3,6.2,11751,0.49,463,NaN,stable,False
4,44.0,19140,0.13,263,2803.0,down,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

***A fixed rule might say***: "Flag any page older than 200 days with position > 10." This fails for three reasons shown in the data:

58.6% of pages with good impressions are still declining — so high traffic alone is not a safe signal. A rule based on impressions volume would miss them.
Position and content age interact non-linearly — a new page at position 12 has a different risk profile than a 3-year-old page at position 12. A Decision Tree can capture this interaction; a single if-statement cannot.
The signals are tangled across 30,000 rows — the combinations of CTR, position, age, and impression volume are too complex to enumerate in hand-written rules without overfitting to edge cases.
ML earns its place here by learning the combination of signals that precedes decline, not just the presence of any single one.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.